# 03 Data Cleaning & Standardization
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Objective:
Clean, transform, handle zero-variance columns, standardize schemas, and export processed datasets to `data/processed/`.


In [2]:
import os
import re
import pandas as pd
import numpy as np

DATA_RAW = "../data/raw"
DATA_PROCESSED = "../data/processed"
os.makedirs(DATA_PROCESSED, exist_ok=True)


In [3]:
# 1. Clean Employee Attrition Dataset
df_attrition_raw = pd.read_csv(os.path.join(DATA_RAW, "employee_attrition.csv"))

# Drop constant zero-variance columns
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours']
df_attrition_clean = df_attrition_raw.drop(columns=cols_to_drop).copy()

def to_snake_case(s):
    s = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', s)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s).lower().strip('_')

df_attrition_clean.columns = [to_snake_case(c) for c in df_attrition_clean.columns]
df_attrition_clean.rename(columns={'employee_number': 'employee_id'}, inplace=True)

df_attrition_clean['attrition_binary'] = (df_attrition_clean['attrition'] == 'Yes').astype(int)
df_attrition_clean['over_time_binary'] = (df_attrition_clean['over_time'] == 'Yes').astype(int)

out_attrition = os.path.join(DATA_PROCESSED, "employee_attrition_processed.csv")
df_attrition_clean.to_csv(out_attrition, index=False)
print(f"Saved: {out_attrition} ({df_attrition_clean.shape[0]} rows, {df_attrition_clean.shape[1]} cols)")


Saved: ../data/processed\employee_attrition_processed.csv (1470 rows, 34 cols)


In [4]:
# 2. Clean Engagement & Performance Dataset
df_perf_raw = pd.read_csv(os.path.join(DATA_RAW, "hr_performance_engagement.csv"))

df_perf_clean = df_perf_raw.copy()
df_perf_clean.columns = [
    'employee_id', 'name', 'department', 'job_role', 'performance_score',
    'kpi_score', 'attendance_pct', 'peer_rating', 'task_completion_pct',
    'work_hours_logged', 'manager_feedback', 'training_hours', 'promotion_eligibility'
]

# Calculate standardized composite Engagement Score (0 - 100)
# Formula: 0.35 * Attendance% + 0.35 * TaskCompletion% + 0.30 * (PeerRating / 5.0 * 100)
df_perf_clean['engagement_score'] = np.round(
    (0.35 * df_perf_clean['attendance_pct']) + 
    (0.35 * df_perf_clean['task_completion_pct']) + 
    (0.30 * (df_perf_clean['peer_rating'] / 5.0 * 100)),
    2
)

out_perf = os.path.join(DATA_PROCESSED, "engagement_processed.csv")
df_perf_clean.to_csv(out_perf, index=False)
print(f"Saved: {out_perf} ({df_perf_clean.shape[0]} rows, {df_perf_clean.shape[1]} cols)")


Saved: ../data/processed\engagement_processed.csv (5000 rows, 14 cols)


In [5]:
# 3. Clean Occupation Master Dataset
df_occ_raw = pd.read_csv(os.path.join(DATA_RAW, "occupation_data.csv"))
df_occ_clean = df_occ_raw.copy()
df_occ_clean.columns = ['soc_code', 'title', 'description']
df_occ_clean['title'] = df_occ_clean['title'].str.strip()
df_occ_clean['description'] = df_occ_clean['description'].str.strip()

out_occ = os.path.join(DATA_PROCESSED, "occupation_master.csv")
df_occ_clean.to_csv(out_occ, index=False)
print(f"Saved: {out_occ} ({df_occ_clean.shape[0]} rows, {df_occ_clean.shape[1]} cols)")


Saved: ../data/processed\occupation_master.csv (1016 rows, 3 cols)


In [6]:
# 4. Clean Essential Skills Dataset
df_ess_raw = pd.read_csv(os.path.join(DATA_RAW, "essential_skills.csv"))

df_ess_im = df_ess_raw[df_ess_raw['Scale ID'] == 'IM'][['O*NET-SOC Code', 'Title', 'Element Name', 'Data Value']].rename(columns={'Data Value': 'importance'})
df_ess_lv = df_ess_raw[df_ess_raw['Scale ID'] == 'LV'][['O*NET-SOC Code', 'Element Name', 'Data Value']].rename(columns={'Data Value': 'level'})

df_ess_clean = pd.merge(df_ess_im, df_ess_lv, on=['O*NET-SOC Code', 'Element Name'], how='inner')
df_ess_clean.columns = ['soc_code', 'title', 'skill_name', 'importance', 'level']
df_ess_clean['skill_name'] = df_ess_clean['skill_name'].str.strip()

out_ess = os.path.join(DATA_PROCESSED, "essential_skills_processed.csv")
df_ess_clean.to_csv(out_ess, index=False)
print(f"Saved: {out_ess} ({df_ess_clean.shape[0]} rows, {df_ess_clean.shape[1]} cols)")


Saved: ../data/processed\essential_skills_processed.csv (9100 rows, 5 cols)


In [7]:
# 5. Clean Software Skills Dataset
df_soft_raw = pd.read_csv(os.path.join(DATA_RAW, "software_skills.csv"))

df_soft_clean = df_soft_raw[['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element Name', 'Hot Technology', 'In Demand']].copy()
df_soft_clean.columns = ['soc_code', 'title', 'skill_name', 'category', 'hot_technology', 'in_demand']
df_soft_clean['skill_name'] = df_soft_clean['skill_name'].str.strip()
df_soft_clean['hot_technology'] = (df_soft_clean['hot_technology'] == 'Y').astype(int)
df_soft_clean['in_demand'] = (df_soft_clean['in_demand'] == 'Y').astype(int)
df_soft_clean.drop_duplicates(subset=['soc_code', 'skill_name'], inplace=True)

out_soft = os.path.join(DATA_PROCESSED, "software_skills_processed.csv")
df_soft_clean.to_csv(out_soft, index=False)
print(f"Saved: {out_soft} ({df_soft_clean.shape[0]} rows, {df_soft_clean.shape[1]} cols)")


Saved: ../data/processed\software_skills_processed.csv (31821 rows, 6 cols)
